# 03. Instance-based model: K-Nearest Neighbours
**Run on:** Laptop A · **Approx. runtime:** ~6-10 min (prediction is slow) on a 4-core laptop


**Protocol (same for every model):**
- Each row is one *(GPS point, candidate arc)* pair. The label is 1 if the arc is the point's reference arc.
- **5-fold GroupKFold by trip** on the 78 usable training trips. Each fold fits on a 300k-row sample of points (KNN uses 100k) and is scored on all candidates of the held-out trips.
- The model is then **refit on the training split** and scored on the **10 student-test trips**.
- **Metrics:**
  - macro F1 / precision / recall: candidate level, threshold 0.5
  - log loss
  - **Top-1 accuracy**: the share of GPS points whose highest-scoring candidate is the reference arc
  - continuity
- The **HMM/Viterbi** decoder (parameters in `artifacts/hmm_params.json`) is then run on this model's probabilities to compare *classifier only* with *classifier + HMM*.


**Before running:** the dataset must be in `data/train/<id>/` and `data/student_test/<id>/`, and `pip install -r requirements.txt` done. Run all cells top to bottom (Kernel → Restart & Run All). Every number and plot below is computed when you run it.

In [ ]:
import os, sys, warnings
# run from notebooks/ or from the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160)
from pathsnap import nb
from pathsnap.models import make_model
from pathsnap.config import CFG
SEED = CFG["seed"]  # 42: every model and data split uses this seed
print("repo root:", ROOT)

In [ ]:
tr, tr_pts, te, te_pts = nb.load_data()

## KNN


**What it does:** no training. For a new (point, candidate) row it finds the 25 most similar training rows (Euclidean distance on standardised features) and takes a distance-weighted vote.
- **Why tried:** a non-parametric baseline; similar geometric situations should have similar answers.
- **Limitations:**
  - prediction is slow, since it compares against every stored row; we fit on 100k rows to keep it tractable
  - sensitive to feature scaling
  - probabilities are coarse, since they come from votes
- **Hyperparameters:** k = 25, weights = distance, metric = Minkowski (p = 2).
- **Objective:** none (lazy learner).

In [ ]:
make_knn = lambda: make_model("knn", SEED)
print(make_knn())
cv_knn = nb.run_cv(make_knn, tr, tr_pts, max_rows=100000)
cv_knn.round(4)

In [ ]:
model_knn, train_knn, test_knn, p_knn = nb.fit_and_evaluate(make_knn, tr, tr_pts, te, te_pts, max_rows=100000)
print(f"student-test continuity (classifier only): {test_knn['continuity']:.4f} | inference {test_knn['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("KNN", cv_knn, train_knn, test_knn)

In [ ]:
imp_knn = nb.plot_importance(model_knn, te, "KNN")

In [ ]:
hmm_knn = nb.hmm_compare(model_knn, te, "KNN")
nb.plot_hmm(hmm_knn, "KNN: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("knn", cv_knn, train_knn, test_knn,
                {"hmm_mean": hmm_knn.drop(columns=["trip", "model"]).mean().to_dict()})

## Comparison of the models in this notebook

In [ ]:
nb.plot_cv_folds({"KNN": cv_knn})
nb.plot_overfit({"KNN": (train_knn, cv_knn, test_knn)})
nb.plot_confusion({"KNN": test_knn["confusion"]})
nb.plot_calibration(te.label.to_numpy(), {"KNN": p_knn})

In [ ]:
rows = []
for name, cv, t, h in [("KNN", cv_knn, test_knn, hmm_knn)]:
    rows.append({"model": name, "CV macro F1": cv.macro_f1.mean(), "CV std": cv.macro_f1.std(),
                 "test Top-1": t["top1"], "test macro F1": t["macro_f1"], "test log loss": t["log_loss"],
                 "test continuity": t["continuity"], "Top-1 + HMM (trip mean)": h.hmm_top1.mean(),
                 "route F1 + HMM (trip mean)": h.hmm_route_f1.mean()})
pd.DataFrame(rows).set_index("model").round(4)